# ECON 5140: Applied Econometrics
## Homework 3: Causal Inference — Potential Outcomes & Matching

**Covers:** Lesson 7 (Potential Outcomes & DAGs) & Lesson 9 (Matching for Causal Inference)

---

# PART 1: ANALYTICAL PROBLEMS

### Problem 1: Potential Outcomes and Causal Estimands

Consider the following data from 8 individuals. D = 1 if they received job training; D = 0 if not. Y(1) and Y(0) are potential outcomes (earnings in thousands).

| i | Y(1) | Y(0) | D | Y_obs |
|---|------|------|---|-------|
| 1 | 45   | 35   | 1 | 45    |
| 2 | 50   | 42   | 1 | 50    |
| 3 | 38   | 40   | 0 | 40    |
| 4 | 55   | 48   | 1 | 55    |
| 5 | 42   | 38   | 0 | 38    |
| 6 | 48   | 45   | 1 | 48    |
| 7 | 40   | 44   | 0 | 44    |
| 8 | 52   | 46   | 0 | 46    |

**Questions:**

a) Calculate ATE, ATT, and ATU.

b) Calculate the Simple Difference in Outcomes (SDO) = E[Y|D=1] − E[Y|D=0]. Is SDO equal to ATE? Why or why not?

c) Decompose SDO into ATE + selection bias + HTE bias. Verify: SDO = ATE + (E[Y(0)|D=1] − E[Y(0)|D=0]) + p×(ATT − ATU).

d) If treatment were randomly assigned, what would happen to the SDO? Explain.

### Problem 2: DAGs — Confounding and Colliders

**Part (a)** Consider the DAG: D ← X → Y and D → Y, where D = treatment, Y = outcome, X = observed covariate.

- What is X called? Why does a naive comparison of D and Y (without controlling for X) produce bias?
- What adjustment would you make to estimate the causal effect of D on Y?

**Part (b)** Consider the DAG: D → X ← Y, where X is a collider.

- Are D and Y independent in the population? Explain.
- What happens if we condition on X? Why is this problematic?
- Give a real-world example of a collider (similar to the "admitted to elite university" example from lecture).

### Problem 3: Exact vs Approximate Matching

a) What is exact matching? When does it work well? What is the "curse of dimensionality" and why does it limit exact matching?

b) What is approximate (distance-based) matching? Name two distance metrics. How does nearest-neighbor matching work?

### Problem 4: Inverse Probability Weighting (IPW) — Numerical Example

Suppose we observe the following data. D = 1 if received job training, D = 0 if not. Y = earnings (thousands). The propensity score e(X) = Pr(D=1|X) has been estimated for each unit.

| i | D | Y  | e(X) |
|---|---|-----|------|
| 1 | 1 | 52 | 0.6  |
| 2 | 1 | 48 | 0.8  |
| 3 | 0 | 42 | 0.3  |
| 4 | 0 | 38 | 0.2  |
| 5 | 1 | 55 | 0.7  |
| 6 | 0 | 45 | 0.4  |

**Questions:**

a) **ATE weights:** For each unit i, the IPW weight for ATE is w_i = D_i/e(X_i) + (1−D_i)/(1−e(X_i)). Compute the weight for each of the 6 units. Fill in the table:

| i | D | Y  | e(X) | w_ATE |
|---|---|-----|------|-------|
| 1 | 1 | 52 | 0.6  | ?     |
| 2 | 1 | 48 | 0.8  | ?     |
| ... |   |    |      |       |

b) **ATT weights:** For ATT, treated units get weight 1; control units get weight w_i = e(X_i)/(1−e(X_i)). Compute the ATT weight for each unit. Fill in the table:

| i | D | Y  | e(X) | w_ATT |
|---|---|-----|------|-------|
| 1 | 1 | 52 | 0.6  | ?     |
| ... |   |    |      |       |

c) **ATE estimate:** Compute the IPW estimator for ATE:


d) **ATT estimate:** Compute the IPW estimator for ATT:



e) **Interpretation:** Which unit has the largest ATE weight? Why? (Hint: "Units unlikely to receive their observed treatment get larger weight.")

# PART 2: CODING PROBLEMS

## Problem 5: Propensity Score Model (Coding)

**Context (tech company):** A SaaS company rolled out an AI assistant feature in a phased rollout. Power users and higher-tier customers were more likely to get early access. You want to estimate the causal effect of the feature on **weekly active minutes**.

**Variables:**
- **D** = 1 if user had access to AI assistant, 0 otherwise
- **Y** = weekly_active_minutes (outcome)
- **plan_tier**: 0=free, 1=pro, 2=enterprise
- **tenure_days**: days since signup (0–365)
- **prior_sessions**: sessions in past 30 days before launch
- **platform**: 0=web, 1=mobile
- **signup_cohort**: 0=Q1, 1=Q2, 2=Q3 (quarter of signup)

**True causal effect:** +15 minutes per week. Treatment is *not* random—selection on observables.

**Your tasks:**

1. **Use the simulated data** below (or generate your own with the same DGP).

2. **Estimate the propensity score** e(X) = Pr(D=1|X) using logistic regression of D on all covariates. Print the model summary. Which covariates most increase the probability of treatment?

3. **Add fitted propensity scores** to the dataset.

4. **Check overlap:** Plot propensity score distributions for treated vs control (KDE or histogram). Is overlap sufficient?

5. **Compute ATE via IPW.** Compare naive ATE, IPW ATE, and true effect (15). Briefly interpret.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(123)

# Simulated data: AI assistant feature rollout (phased, not random)
n = 600
plan_tier = np.random.choice([0, 1, 2], n, p=[0.5, 0.35, 0.15])
tenure_days = np.random.randint(0, 366, n)
prior_sessions = np.random.poisson(12, n) + np.random.randint(0, 20, n)
platform = np.random.choice([0, 1], n, p=[0.6, 0.4])
signup_cohort = np.random.choice([0, 1, 2], n, p=[0.4, 0.35, 0.25])

# Treatment: higher tier, tenure, prior sessions, mobile → more likely to get feature
logit_p = -2.0 + 0.5*plan_tier + 0.003*tenure_days + 0.02*prior_sessions + 0.4*platform + 0.2*signup_cohort
ps_true = 1 / (1 + np.exp(-np.clip(logit_p, -10, 10)))
D = (np.random.uniform(0, 1, n) < ps_true).astype(int)

# Outcome: weekly_active_minutes. True effect of feature = +15 min
Y = 80 + 15*D + 8*plan_tier + 0.05*tenure_days + 0.8*prior_sessions + 5*platform + 3*signup_cohort + np.random.normal(0, 12, n)
Y = np.maximum(Y, 0)

df = pd.DataFrame({
    'plan_tier': plan_tier, 'tenure_days': tenure_days, 'prior_sessions': prior_sessions,
    'platform': platform, 'signup_cohort': signup_cohort, 'D': D, 'weekly_active_minutes': Y
})

print("Dataset: AI assistant feature rollout")
print(f"  n = {len(df)}, Treated: {D.sum()}, Control: {n - D.sum()}")
print(f"  True causal effect: +15 min/week")
print(df.head(10))

Dataset: AI assistant feature rollout
  n = 600, Treated: 254, Control: 346
  True causal effect: +15 min/week
   plan_tier  tenure_days  prior_sessions  platform  signup_cohort  D  \
0          1          231              16         1              2  0   
1          0          276              28         0              2  0   
2          0          345              21         0              1  1   
3          1          353              23         0              1  1   
4          1           39              15         0              0  0   
5          0          205              28         0              0  0   
6          2          298              17         0              0  0   
7          1           58              31         0              0  1   
8          0           48              14         1              2  0   
9          0          174               9         0              2  0   

   weekly_active_minutes  
0             132.725826  
1             114.294894  
2   

In [4]:
# YOUR CODE for Problem 5
# Covariates: plan_tier, tenure_days, prior_sessions, platform, signup_cohort
# Outcome: weekly_active_minutes. True effect = 15
# 1. Fit logistic regression: D ~ covariates
# 2. Add df['ps'] = fitted propensity scores
# 3. Plot overlap (KDE or histogram)
# 4. Compute ATE via IPW (clip ps to avoid extreme weights)
# 5. Compare naive ATE, IPW ATE, true effect (15)

